In [1]:
!pip install transformers datasets accelerate scikit-learn pandas numpy -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_PATH = "/content/drive/MyDrive/neurotalk_project(MULTI)"
MODEL_PATH = f"{PROJECT_PATH}/model"

os.makedirs(MODEL_PATH, exist_ok=True)

print("Project folder ready:", MODEL_PATH)

Project folder ready: /content/drive/MyDrive/neurotalk_project(MULTI)/model


In [4]:
import pandas as pd
import torch
import numpy as np

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

In [5]:
EMOTIONS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

ID_TO_LABEL = {i: label for i, label in enumerate(EMOTIONS)}

In [6]:
EMOTION_MAPPING = {
    "joy": "joy", "amusement": "amusement", "love": "love", "pride": "pride",
    "sadness": "sadness", "anger": "anger", "fear": "fear", "surprise": "surprise",
    "curiosity": "curiosity", "neutral": "neutral", "excitement": "excitement",
    "relief": "relief", "annoyance": "annoyance", "nervousness": "nervousness",
    "confusion": "confusion"
}

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
if os.path.exists("GoEmotions (1).csv"):
    df_go = pd.read_csv("GoEmotions (1).csv")
elif os.path.exists("GoEmotions.csv"):
    df_go = pd.read_csv("GoEmotions.csv")
else:
    raise FileNotFoundError("GoEmotions.csv not found!")

df_go = df_go[['text'] + EMOTIONS].dropna()

print("GoEmotions size:", len(df_go))

In [ ]:
if os.path.exists("emoji_emotion_dataset.csv"):
    df_emoji = pd.read_csv("emoji_emotion_dataset.csv").dropna()

    augmented_rows = []
    OVERSAMPLE = 20

    for _, row in df_emoji.iterrows():
        emoji = row['emoji']
        meanings = row['meanings']
        emotion = row['emotion']

        mapped = EMOTION_MAPPING.get(emotion, emotion)

        if mapped in EMOTIONS:
            for _ in range(OVERSAMPLE):
                augmented_rows.append({'text': emoji, 'label': mapped})
                augmented_rows.append({'text': f"{emoji} {meanings}", 'label': mapped})

    df_aug = pd.DataFrame(augmented_rows)

    zeros = np.zeros((len(df_aug), len(EMOTIONS)), dtype=int)
    df_emoji_final = pd.DataFrame(zeros, columns=EMOTIONS)
    df_emoji_final['text'] = df_aug['text'].values

    for idx, row in df_aug.iterrows():
        df_emoji_final.at[idx, row['label']] = 1

    df_final = pd.concat([df_go, df_emoji_final], ignore_index=True)

else:
    print("No emoji dataset found")
    df_final = df_go

print("Final dataset size:", len(df_final))

In [ ]:
texts = df_final['text'].tolist()
labels = df_final[EMOTIONS].values.tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=42
)

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Add emoji tokens
if 'df_emoji' in locals():
    unique_emojis = df_emoji['emoji'].unique().tolist()
    tokenizer.add_tokens(unique_emojis)

In [ ]:
class GoEmotionsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]).float()
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = GoEmotionsDataset(train_texts, train_labels, tokenizer)
val_dataset = GoEmotionsDataset(val_texts, val_labels, tokenizer)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(EMOTIONS),
    problem_type="multi_label_classification"
)

model.resize_token_embeddings(len(tokenizer))

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=50
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("✅ Model saved to Drive:", MODEL_PATH)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)

In [ ]:
text = "I am happy but also nervous 😊"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)

probs = torch.sigmoid(outputs.logits).detach().numpy()[0]

print("\nTop Emotions:")
for i, p in enumerate(probs):
    if p > 0.3:
        print(f"{EMOTIONS[i]} → {round(p*100,2)}%")